# Small CI benchmark: offline validation
Validates archived inputs and recorded output; never runs SAG or changes verdicts.

In [ ]:
from pathlib import Path
import json, hashlib, sys, tempfile, tarfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/sag').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'src'))
from scripts.small_ci_bench import read_official
BENCH = ROOT / 'benchmarks/small-ci-10-v1'
manifest = json.loads((BENCH / 'manifest.json').read_text())
assert hashlib.sha256((BENCH / 'manifest.json').read_bytes()).hexdigest() == (BENCH / 'manifest.sha256').read_text().strip()
assert len(manifest['projects']) == len({p['repo'] for p in manifest['projects']}) == 10


In [ ]:
totals = {'modules': 0, 'reported': 0, 'passed': 0, 'red': 0, 'skipped': 0}
archive = BENCH / 'official-ci-evidence.tar.gz'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == (BENCH / 'official-ci-evidence.sha256').read_text().split()[0]
with tempfile.TemporaryDirectory(prefix='small-ci-validation-') as folder:
    with tarfile.open(archive) as tar:
        tar.extractall(folder, filter='data')
    for project in manifest['projects']:
        target, counts = read_official(project, Path(folder) / project['seat'])
        assert target.cells[0].grade == 'A'
        target_file = BENCH / 'targets' / (project['seat'] + '.json')
        assert hashlib.sha256(target_file.read_bytes()).hexdigest() == project['target_sha256']
        assert target.model_dump(mode='json') == json.loads(target_file.read_text())
        assert counts == project['official']
        assert counts['reported'] == counts['passed'] + counts['red'] + counts['skipped']
        assert counts['build_modules'] > 0 and counts['reported'] > 0
        totals['modules'] += counts['build_modules']
        for key in ('reported', 'passed', 'red', 'skipped'): totals[key] += counts[key]
assert totals == {'modules': 42, 'reported': 7872, 'passed': 7850, 'red': 0, 'skipped': 22}
print(totals)


In [ ]:
from fractions import Fraction
comparison = ROOT / 'logs/small-ci-bench-10-v1-20260909/report-v3/comparison.json'
if comparison.exists():
    rows = json.loads(comparison.read_text())['rows']
    assert len(rows) == 10
    for row in rows:
        for counts in (row['sag_counts'], row.get('archive_counts')):
            if counts is not None:
                assert counts['reported'] == counts['passed'] + counts['red'] + counts['skipped']
        if row['alpha'] is not None:
            a, b, c = row['alpha'], row['alpha_build'], row['alpha_test']
            assert b is not None and c is not None
            fraction = lambda p: Fraction(p['numerator'], p['denominator'])
            assert fraction(a) == min(fraction(b), fraction(c))
    print('Terminal rows:', sum(r['status'] == 'completed' for r in rows))
